# Bill&Meal: Receipt-to-Recipe Training

This notebook trains a student vision-language model using the Bill&Meal package.
All logic lives in the `bill_and_meal` package — this notebook just calls it.

## 1. Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone repo and install package
# Replace with your repo URL
!git clone https://github.com/YOUR_USER/bill_and_meal.git /content/bill_and_meal
%cd /content/bill_and_meal
!pip install -e . -q

In [ ]:
# Set API keys
import os
os.environ['ANTHROPIC_API_KEY'] = 'your-key-here'  # or use Colab secrets
os.environ['WANDB_API_KEY'] = 'your-key-here'

## 2. Verify Data

In [ ]:
from pathlib import Path
from bill_and_meal.config import load_config

config = load_config()
receipts_dir = Path(config['data']['receipts_dir'])
labeled_path = Path(config['data']['labeled_path'])

receipt_count = len(list(receipts_dir.glob('*'))) if receipts_dir.exists() else 0
labeled_count = sum(1 for _ in open(labeled_path)) if labeled_path.exists() else 0

print(f'Receipts: {receipt_count}')
print(f'Labeled pairs: {labeled_count}')

## 3. Train

In [ ]:
from bill_and_meal.train import run_training

run_training(config)

## 4. Evaluate

In [ ]:
import json
from bill_and_meal.evaluate import ingredient_coverage

with open(config['data']['labeled_path']) as f:
    records = [json.loads(line) for line in f if line.strip()]

for r in records[:5]:
    cov = ingredient_coverage(r['ingredients'], r['teacher_output'])
    print(f"{r['id']}: coverage={cov:.1%}")

## 5. Export

In [ ]:
import shutil
from pathlib import Path

output_dir = Path(config['training']['output_dir'])
if output_dir.exists():
    print(f'Model saved at: {output_dir}')
    print('Files:', list(output_dir.iterdir()))
else:
    print('No model output found. Did training complete?')